# Cross-Domain Evaluation: Fact-Grounding Classifier

This notebook independently loads a persisted Kaggle checkpoint Dataset and prepared evaluation Dataset. It contains no training or threshold tuning. The fixed answerable threshold is 0.5. HotpotQA labels are constructed from its annotated supporting facts using paired evidence sets; SciFact verdict labels and FiQA relevance judgments are not treated as answerability labels. Unsupported datasets are explicitly reported as not evaluated.


In [ ]:
import sys,subprocess,json,time,random
from pathlib import Path
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1','sentencepiece','protobuf','scikit-learn'])
import numpy as np,pandas as pd,torch,transformers
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from sklearn.metrics import precision_score,recall_score,f1_score,accuracy_score,balanced_accuracy_score,roc_auc_score,confusion_matrix
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available()
print({'python':sys.version,'torch':torch.__version__,'transformers':transformers.__version__,'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})

In [ ]:
def input_dir(fragment):
    hits=[p for p in Path('/kaggle/input').iterdir() if fragment in p.name.lower()]
    if not hits: raise FileNotFoundError(f'Attach Kaggle input dataset containing {fragment!r}')
    return hits[0]
ckpt_root=input_dir('fact-grounding-checkpoint'); eval_root=input_dir('fact-grounding-eval-data')
configs=list(ckpt_root.rglob('config.json')); ckpt=next((p.parent for p in configs if (p.parent/'model.safetensors').exists() or (p.parent/'pytorch_model.bin').exists()),None)
if ckpt is None: raise FileNotFoundError('Model checkpoint weights not found')
tokenizer=AutoTokenizer.from_pretrained(ckpt); model=AutoModelForSequenceClassification.from_pretrained(ckpt).cuda().eval()
assert model.config.label2id.get('ANSWERABLE')==1 and model.config.label2id.get('NOT-ANSWERABLE')==0
examples_file=next(eval_root.rglob('prepared_eval_examples.jsonl')); manifest_file=next(eval_root.rglob('dataset_manifest.json'))
examples=[json.loads(s) for s in examples_file.read_text(encoding='utf-8').splitlines() if s.strip()]
manifest=json.loads(manifest_file.read_text(encoding='utf-8'))
need={'id','dataset','question','passages','label','label_source','construction_method','original_document_ids'}
assert all(need<=set(e) and e['label'] in (0,1) for e in examples)
df=pd.DataFrame(examples); print(df.groupby(['dataset','label']).size().rename('n').to_frame()); print(json.dumps(manifest,indent=2))

## Fixed-threshold evaluation

Report positive-class precision, recall, F1, macro-F1, balanced accuracy, AUROC, 15-bin ECE, Brier score, confusion matrix, and inference latency. AUROC is omitted when only one class is present. Errors are saved per example.


In [ ]:
def ece(y,p,bins=15):
    edges=np.linspace(0,1,bins+1); out=0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        m=(p>=lo)&((p<hi) if hi<1 else (p<=hi))
        if m.any(): out+=m.mean()*abs(float(y[m].mean())-float(p[m].mean()))
    return float(out)
rows=[]; pred_rows=[]
for name,g in df.groupby('dataset',sort=True):
    texts=[]
    for r in g.itertuples():
        passage_text='\\n'.join(r.passages)[:1500]
        texts.append(f"Sub-question: {r.question}\\n\\nPassages:\\n{passage_text}")
    y=g.label.to_numpy(dtype=int); p=[]; t=time.perf_counter()
    for i in range(0,len(texts),16):
        x=tokenizer(texts[i:i+16],max_length=512,padding='max_length',truncation=True,return_tensors='pt').to('cuda')
        with torch.no_grad(): p.extend(torch.softmax(model(**x).logits.float(),dim=-1)[:,1].cpu().numpy().tolist())
    elapsed=time.perf_counter()-t; p=np.asarray(p); yh=(p>=.5).astype(int)
    rows.append({'dataset':name,'n':len(y),'n_positive':int(y.sum()),'n_negative':int((1-y).sum()),'threshold':.5,'accuracy':accuracy_score(y,yh),'precision':precision_score(y,yh,zero_division=0),'recall':recall_score(y,yh,zero_division=0),'f1':f1_score(y,yh,zero_division=0),'macro_f1':f1_score(y,yh,average='macro',zero_division=0),'balanced_accuracy':balanced_accuracy_score(y,yh),'auroc':roc_auc_score(y,p) if len(np.unique(y))==2 else None,'ece_15':ece(y,p),'brier_score':float(np.mean((p-y)**2)),'confusion_matrix_0_1':confusion_matrix(y,yh,labels=[0,1]).tolist(),'inference_seconds':elapsed,'latency_ms_per_example':elapsed*1000/len(y),'status':'evaluated'})
    for e,prob,hat in zip(g.to_dict('records'),p,yh): pred_rows.append({**e,'prob_answerable':float(prob),'prediction':int(hat),'correct':int(hat==e['label'])})
for name in ['MuSiQue','HotpotQA','SciFact','FiQA-2018']:
    if not any(r['dataset']==name for r in rows): rows.append({'dataset':name,'n':0,'status':'not_evaluated','reason':manifest.get('evaluation_data',{}).get('not_evaluated',{}).get(name,'No compatible labeled examples attached.'),'threshold':.5})
out=Path('/kaggle/working/results'); out.mkdir(parents=True,exist_ok=True)
pd.DataFrame(rows).to_csv(out/'cross_domain_metrics.csv',index=False); pd.DataFrame(pred_rows).to_csv(out/'predictions.csv',index=False)


## Question-only ablation and failure cases

Question-only is an intentionally out-of-distribution diagnostic. It does not change weights or threshold. False positives and false negatives are exported for review.


In [ ]:
abl=[]
for name,g in df.groupby('dataset'):
    texts=[f'Sub-question: {q}\\n\\nPassages:\\n' for q in g.question]
    y=g.label.to_numpy(dtype=int); p=[]
    for i in range(0,len(texts),16):
        x=tokenizer(texts[i:i+16],max_length=512,padding='max_length',truncation=True,return_tensors='pt').to('cuda')
        with torch.no_grad(): p.extend(torch.softmax(model(**x).logits.float(),dim=-1)[:,1].cpu().numpy().tolist())
    yh=(np.asarray(p)>=.5).astype(int)
    abl.append({'dataset':name,'condition':'question_only_ood','n':len(y),'accuracy':accuracy_score(y,yh),'f1':f1_score(y,yh,zero_division=0),'macro_f1':f1_score(y,yh,average='macro',zero_division=0)})
pd.DataFrame(abl).to_csv(out/'question_only_ablation.csv',index=False)
pred=pd.DataFrame(pred_rows)
if len(pred):
    fail=pred[pred.correct==0].copy(); fail['error_type']=np.where((fail.label==0)&(fail.prediction==1),'false_positive_missing_evidence','false_negative_sufficient_evidence')
else: fail=pred
fail.to_csv(out/'failure_cases.csv',index=False)
print(pd.DataFrame(rows).to_string(index=False)); print('failure_cases=',len(fail))